In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
import os
import pynbody
import pynbody.plot.sph as sph
import pyarrow

SMALLSIZE = 9
NORMALSIZE = 9
LARGESIZE = 9

params = {
    "font.family":"serif",
    "mathtext.fontset":"stix",
    "font.size": SMALLSIZE,
    "axes.titlesize" : SMALLSIZE,
    "axes.labelsize" : NORMALSIZE,
    'xtick.labelsize': SMALLSIZE,
    'ytick.labelsize': SMALLSIZE,      
    'legend.fontsize': SMALLSIZE,  
    'figure.titlesize': LARGESIZE,
    'pgf.texsystem' : "pdflatex"
}
matplotlib.rcParams.update(params)

isob_dir = "../data/IsoB_dt10_all"
snap_names = [f"GLX.000{10*(j):03.0f}" for j in range(1, 62)] + [f"GLX.0{10*j:04.0f}" for j in range(62, 101)]


In [ ]:
for i, snap in enumerate(snap_names):
    sim = pynbody.load(f"{isob_dir}/{snap}")
    sim.physical_units()
    pynbody.analysis.angmom.faceon(sim)
    df = pd.DataFrame(dict(
        iord = sim.s['iord'],
        x = sim.s['x'],
        y = sim.s['y'],
        z = sim.s['z'],
        mass = sim.s['mass']
    ))
    text = snap.split(".")
    df.to_csv(f"../data/IsoB_dt10_stars/{text[0]}.0{text[1]}.csv")
    print(i)

In [46]:
pq_names = [f"GLX.00{10*(j):04.0f}" for j in range(1, 101)]

In [ ]:
for i in pq_names:
    df = pd.read_csv(f"../data/IsoB_dt10_stars/{i}.csv")
    df['rz'] = np.sqrt(df['x']**2 + df['y']**2)
    df['phiz'] = np.atan2(df['y'], df['x'])
    df[['iord', 'x', 'y', 'z', 'rz', 'phiz', 'mass']].to_parquet(f"../data/stars/{i}.parquet", engine="pyarrow", compression='snappy')
    print(i)

In [ ]:
nrows = 10
ncols = 10
fig, axes = plt.subplots(nrows=nrows, ncols=ncols)
axes = axes.ravel()
fig.set_size_inches(4*ncols, 3*nrows)

for i, pq in enumerate(pq_names):
    df = pd.read_parquet(f"../data/stars/{pq}.parquet")
    df = df[df['rz'] < 10]
    ax = axes[i]
    ax.hist2d(df['x'], df['y'], bins=50, norm='log', cmap='inferno')
    print(i)

In [ ]:
fig.savefig("../figures/isob_evolution.png", format='png', bbox_inches='tight', dpi=50)